# Production pipeline

В данном ноутбуке реализован автоматизированный пайплайн прогнозирования сальдо ликвидности на следующий рабочий день.

В качестве финальной модели используется SARIMA. По точности на тестовой выборке CatBoost с отбором признаков незначительно превосходит SARIMA (MAE 0.181 против 0.185), однако SARIMA выбрана как финальная модель из-за существенных практических преимуществ: минимальные требования к подготовке данных, отсутствие необходимости в генерации признаков, простота настройки и интерпретируемость результатов.

Пайплайн выполняет следующие действия:

1. Загружает актуальные данные.
2. Исключает выходные дни.
3. Формирует временной ряд сальдо.
4. Заменяет выбросы линейной интерполяцией.
5. Проверяет необходимость переобучения модели.
6. Контролирует наличие разладки временного ряда методом CUSUM.
7. При необходимости автоматически переобучает модель.
8. Формирует прогноз на следующий рабочий день.
9. Сохраняет результат прогноза.

Период регулярного переобучения выбран равным 5 рабочим дням. Такой выбор обусловлен выявленной недельной сезонностью временного ряда и сезонным параметром SARIMA `s = 5`. Таким образом, модель обновляется после поступления полного нового недельного цикла наблюдений.

Модуль обнаружения разладок реализован методом CUSUM.

В ходе исследования установлено, что выявленные точки в основном соответствуют локальным выбросам и не приводят к существенному изменению структуры ряда. Поэтому модуль обнаружения разладок сохранён как диагностический инструмент мониторинга, а основным триггером переобучения выбран временной критерий — 5 рабочих дней.

In [1]:
from data_loader import load_data
from production_pipeline import run_production_pipeline

In [2]:
df = load_data(file_name="Project 1_2024.xlsx", sheet_name="Data")

In [3]:
forecast_result, model, metadata, drift_info = run_production_pipeline(
    df=df,
    model_path="models/best_sarima.pkl",
    metadata_path="models/metadata.pkl",
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 5),
    retrain_period=5,
    use_drift_control=False,
    use_outlier_processing=True
)

forecast_result

,ForecastDate,Prediction,Model,Retrained,RetrainReason,TrainEndDate,TrainSize,OutliersProcessed,OutliersPercent
0,2021-04-01,-0.398946,"SARIMA(1, 0, 1)x(1, 0, 1, 5)",False,Переобучение не требуется,2021-03-31,1103,60,5.43971


In [4]:
forecast_result.to_excel("reports/next_day_forecast.xlsx", index=False)